# পাঠ ০১ - এআই এজেন্টের পরিচিতি

**শুরুদের জন্য এআই এজেন্টস** কোর্সের প্রথম পাঠে আপনাকে স্বাগতম!

একটি **এআই এজেন্ট** হল একটি প্রোগ্রাম যা একটি বৃহৎ ভাষা মডেল (LLM) কে তার যুক্তি ইঞ্জিন হিসেবে ব্যবহার করে এবং বাস্তব জগতে *কার্য* করতে পারে — API কল করা, ডেটাবেস জিজ্ঞাসা করা, বা কোড চালানো — একটি ব্যবহারকারীর পক্ষে একটি লক্ষ্য অর্জনের জন্য।

এই নোটবুকে আপনি আপনার প্রথম এজেন্ট তৈরি করবেন: একটি **ট্রাভেল এজেন্ট** যা ছুটির গন্তব্য সুপারিশ করে। পাশাপাশি আপনি শিখবেন কিভাবে:

১. **Microsoft Agent Framework** ব্যবহার করে Microsoft Foundry Agent Service-এ সংযোগ করা।
২. এজেন্টকে একটি **টুল** দেওয়া — একটি সাধারণ পাইথন ফাংশন যেটি এটি কল করতে পারে।
৩. এজেন্ট চালানো এবং এর প্রতিক্রিয়া পরীক্ষা করা।
৪. এজেন্টের প্রতিক্রিয়া টোকেন-বাই-টোকেন স্ট্রিম করা।


## সেটআপ

এই নোটবুক চালানোর আগে, নিশ্চিত করুন আপনার কাছে আছে:

1. **একটি Microsoft Foundry প্রকল্প** যা একটি ডিপ্লয় করা চ্যাট মডেল রয়েছে (যেমন `gpt-5-mini`)।
2. **Azure CLI দিয়ে লগ ইন করেছেন** — আপনার টার্মিনালে `az login` কমান্ড চালান।
3. **প্রয়োজনীয় পরিবেশ চলকগুলি সেট করেছেন:**
   - `AZURE_AI_PROJECT_ENDPOINT` — আপনার Microsoft Foundry প্রকল্পের এন্ডপয়েন্ট।
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — আপনার ডিপ্লয় করা মডেলের নাম।

নিচের সেলটি আপনার দরকারি পাইথন প্যাকেজগুলি ইনস্টল করে।


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## আপনার প্রথম এজেন্ট তৈরি করা

একটি এজেন্টের জন্য দুটি জিনিস প্রয়োজন:

- **নির্দেশনা** যা তাকে বলে *সে কে* এবং *কিভাবে আচরণ করবে* (একটি সিস্টেম প্রম্পট)।
- **টুলস** — পাইথন ফাংশন যা `@tool` দিয়ে সজ্জিত, যেগুলো এজেন্ট তথ্য সংগ্রহ করতে বা কার্য সম্পাদনের জন্য কল করতে পারে।

নিচে আমরা একটি সাধারণ টুল সংজ্ঞায়িত করেছি যা জনপ্রিয় ছুটির গন্তব্যের একটি তালিকা প্রদান করে। ব্যবহারকারী যখন ভ্রমণের সুপারিশ চান, তখন এজেন্ট এই টুলটি ব্যবহার করবে।


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## স্ট্রিমিং উত্তরসমূহ

একটি আরও ইন্টারেক্টিভ অভিজ্ঞতার জন্য আপনি এজেন্টের উত্তর **স্ট্রিমিং** করতে পারেন। সম্পূর্ণ প্রতিক্রিয়ার জন্য অপেক্ষা করার বদলে, এজেন্ট যতই টেক্সট তৈরি করে তা অংশে অংশে প্রদান করে। এটি বিশেষত চ্যাট ইন্টারফেসে উপকারী যেখানে আপনি রিয়েল টাইমে আউটপুট প্রদর্শন করতে চান।


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## সারসংক্ষেপ

এই পাঠে আপনি শিখেছিলেন:

- **একটি প্রোভাইডার তৈরি করতে** যা `FoundryChatClient` এর মাধ্যমে Microsoft Foundry Agent Service এর সাথে সংযুক্ত হয়।
- **`@tool` ডেকোরেটর ব্যবহার করে একটি টুল সংজ্ঞায়িত করতে** যাতে এজেন্ট আপনার পাইথন ফাংশনগুলি কল করতে পারে।
- **একটি ব্যবহারকারীর বার্তা নিয়ে এজেন্ট চালাতে এবং তার প্রতিক্রিয়া প্রিন্ট করতে।**
- **রিয়েল-টাইম আউটপুটের জন্য প্রতিক্রিয়া স্ট্রিম করতে।**

পরবর্তী পাঠে আমরা আরও গভীরভাবে এজেন্টিক ফ্রেমওয়ার্কগুলি অন্বেষণ করবো এবং এজেন্টদের আরও শক্তিশালী টুল ও বহু ধাপ যুক্ত যুক্তি ক্ষমতা দেওয়ার কৌশল শিখবো।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**অস্বীকৃতি**:
এই নথিটি AI অনুবাদ পরিষেবা [Co-op Translator](https://github.com/Azure/co-op-translator) ব্যবহার করে অনূদিত হয়েছে। যদিও আমরা শুদ্ধতার জন্য চেষ্টা করি, অনুগ্রহ করে মনে রাখবেন যে স্বয়ংক্রিয় অনুবাদে ত্রুটি বা অসঙ্গতি থাকতে পারে। মূল নথিটি তার স্বভাষায় কর্তৃত্বপূর্ণ উৎস হিসেবে বিবেচিত হওয়া উচিত। গুরুত্বপূর্ণ তথ্যের জন্য পেশাদার মানব অনুবাদ সুপারিশ করা হয়। এই অনুবাদের ব্যবহারে প্রয়োজনীয় ভুল বোঝাবুঝি বা ভুল ব্যাখ্যার জন্য আমরা দায়বদ্ধ নই।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
